**BINARY CLASSIFICATION NEURAL NETWORK**(From Scratch using Numpy):

Here , I have a trained a neural network from scratch using raw python code numpy library.

No deep learning frameworks(tensorflow,pytorch etc) used .  Every beginner can understand this docuementation of how the neural network training pipeline, how internally every framework calculate,compute the numbers.

The flow of a Neural Network training:

DATA  ->  SPLITING(Features,Target)  ->  SCALING DATA  ->  WEIGHTS AND BIAS  ->  FORWARD PROPAGATION  ->  LOSS  ->  BACKWARD PROPAGATION  ->  UPDATE WEIGHTS AND BIAS  ->  **TEST** MODEL

In [ ]:
!pip install numpy scikit-learn

Here , we take the breast cancer datset from sklearn.

The data consist of 569 samples and 30 features.[X(569,30)]. The target is to  find wheather a person is diseased or not.


0 → Malignant (cancerous)




1 → Benign (non-cancerous)


In [61]:
import numpy as np
from sklearn.datasets import load_breast_cancer


data = load_breast_cancer()
X = data.data
y = data.target

print(X.shape)
print(y.shape)





(569, 30)
(569,)


Spliting the data for 80% for training and 20% for testing the data.

We have reshaped the target value 1D->2D array . So that we can perform matrix mutliplication . In Deep learning shapes of the data before and after computation matters the more for further calculataions.

In [62]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)



y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)



We have scaled the values of the features . On scaling the computation makes simple and more accurate  with out scaling .

In [63]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

 n_x -> Sample size 569 rows


 n_h -> Hidden layer Neurons

 n_y -> Output Layer  Neuron(For Binnary Classification we take a single output neuron)

 Weights & Bias:

 w1 and b1 ->  for Input Layer


 w2 and b2 -> for Hidden Layer

In [64]:
def intial_parameters(n_x,n_h,n_y):
  w1 = np.random.randn(n_x,n_h)*0.01
  b1 = np.zeros((1,n_h))

  w2 = np.random.randn(n_h,n_y)*0.01
  b2 = np.zeros((1,n_y))

  return w1,b1,w2,b2


Activation Functions:
 these mattes more in a neural network training it is introduce to learn complex patterns in the data . Introdues non-linearity.

 Sigmoid : output neuron
 Relu : Hidden Layer

In [65]:
def sigmoid(z):
  z = np.clip(z, -500, 500)
  return 1/(1+np.exp(-z))

def Relu(z):
  return np.maximum(0,z)


Forward pass of the neural network here where we train the neural network .
  z=w⋅x+b (mathematical representation)

  z1 -> a1=Relu(z1) -> z2 -> a2=sigmoid(z2)

In [66]:
def forward_propagation(x,w1,b1,w2,b2):
  z1 = np.dot(x,w1)+b1
  a1 = Relu(z1)

  z2 = np.dot(a1,w2)+b2
  a2 = sigmoid(z2)

  return z1, a1, z2, a2

For Binary Classificaton we use Binary Cross Entropy loss function to calculate the loss . The loss function shows us how much error the model makes so bsed on that we can update the weights and bias .

In [67]:
def loss_function(y,y_pred):
  epsilon = 1e-15
  y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # BCE formula
  loss = -np.mean(
        y * np.log(y_pred) +
        (1 - y) * np.log(1 - y_pred)
    )

  return loss

Backward Propagation : This works on the concept Gradient Descent . It calculate the deriavative of the loss and updates the weights and bias .

NOTE:

->The output layer gradient is propostional to how much prediction differs from true label.

-> input*error=gradient




In [68]:
def backward_propagation(x, y, a1, a2, w2, z1):
    m = X.shape[0]

    # ---- Output Layer Gradients ----
    dz2 = a2 - y
    dw2 = (1/m) * np.dot(a1.T, dz2)
    db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)

    # ---- Hidden Layer Gradients ----
    da1 = np.dot(dz2, w2.T)

    # Choose activation derivative:
    # For ReLU:
    dz1 = da1 * (z1 > 0)

    dw1 = (1/m) * np.dot(x.T, dz1)
    db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)

    return dw1, db1, dw2, db2

Learning rate & Updating of weights and bias:

 -> learning rate is for how much step size the model moving for optimal solution.

 ->it updates the model parameters during the backpropagation.

 ->small learning rate -> better solution


In [69]:
def upgrade_parameters(w1, b1, w2, b2, dw1, db1, dw2, db2, learning_rate):
    w1 = w1 - learning_rate*dw1
    b1 = b1 - learning_rate*db1
    w2 = w2 - learning_rate*dw2
    b2 = b2 - learning_rate*db2

    return w1, b1, w2, b2

Here , is the training loop starts we train the model for number itereations so that the model learns patterns more effective and decreses the loss function.

We update the weights and bias for number of epochs and uses the best model .

In [70]:
def train(x,y,n_h,epochs,learning_rate):
  n_x=x.shape[1]
  n_y=1

  w1,b1,w2,b2 = intial_parameters(n_x,n_h,n_y)

  for i in range(epochs):
    z1,a1,z2,a2 = forward_propagation(x,w1,b1,w2,b2)
    loss = loss_function(y,a2)
    dw1, db1, dw2, db2 = backward_propagation(x,y, a1, a2, w2, z1)
    w1, b1, w2, b2 = upgrade_parameters(w1, b1, w2, b2, dw1, db1, dw2, db2, learning_rate)
    if i % 100 == 0:
        print("Epoch: ", i, "Loss: ", loss)

  return w1,b1,w2,b2


Predict the output wheather the output is maligant or beign

In [71]:
def predict(x,w1,b1,w2,b2):
  z1,a1,z2,a2 = forward_propagation(x,w1,b1,w2,b2)
  return (a2>0.5).astype(int)

Test Accuracy of the model

-> We observe the model on training the for number of iteration decreses the loss function and increase the acuuracy

In [72]:
w1,b1,w2,b2 = train(x_train,y_train, n_h=16,epochs=1000,learning_rate=0.01)
y_pred = predict(x_test,w1,b1,w2,b2)
acuuracy = np.mean(y_pred == y_test)
print(acuuracy)

Epoch:  0 Loss:  0.6932079807059264
Epoch:  100 Loss:  0.6809497527625054
Epoch:  200 Loss:  0.6704885929379181
Epoch:  300 Loss:  0.6563076932974125
Epoch:  400 Loss:  0.6260940924396519
Epoch:  500 Loss:  0.5567284274751038
Epoch:  600 Loss:  0.4429126791608869
Epoch:  700 Loss:  0.332783826100851
Epoch:  800 Loss:  0.25673580398395907
Epoch:  900 Loss:  0.20853795736292008
0.9736842105263158


On further we can use optimizers(adam) and can get more acurate output prediction from the model .

-> optimizers are similiar to gradients descent but on using them they are more accurate then the  gradients